[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.5.ipynb)

# **MNPS Job Classification_gpt-4o_Five Pass_POST_PROCESSING v3.1GPT**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> # **Version 7.5.5 Changes**
>   - **Five-Pass Classification**:
>   - Pass 1: Initial LLM classification (attribute-only, full context)
>   - Pass 2: Self-consistency check — LLM reviews its own justification vs classification and corrects mismatches
>   - Pass 3: Manager→Director promotion check for strategic/executive scope
>   - Pass 4: Technician→Skilled Laborer review for physical trade-based work
>   - Pass 5: Manager, Coordinator, Coach review for reclassification
> - **Addresses Technician vs Skilled Laborer confusion**
> - **Addresses Specialist underusage**
> - implements changes—especially the improved discourage_specialist function and the enhanced fifth-pass prompt
> - **Preserves all v7.5.4 logic**: Two-pass, Director pass, validation
> - **Still ignores original job title**, uses full MNPS role/competency context
> - **Same GPT-4o-2024-11-20 model**, rate-limiting, and output saving

# ==== 1) Imports, paths, inputs from v7.1 artifacts ====

In [ ]:
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# If main input file not found, check inside the zip
if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

# ==== 2) Load data and build attribute-only view (ignore title) ====

In [ ]:
# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")

# ==== 3) Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic ====

In [ ]:
# Get MNPS roles from the loaded data
# Handle different possible column names
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    raw_roles = roles_df[role_columns[0]].dropna().astype(str).str.strip().tolist()
else:
    # Fallback to first column
    raw_roles = roles_df.iloc[:, 0].dropna().astype(str).str.strip().tolist()

# Map any roles that should collapse into another canonical role
MAJOR_REPLACEMENTS = {
    # 'Technologist' is not a standalone major group; treat it as a type of Specialist
    'Technologist': 'Specialist',
}

# Roles that should never be offered as major groups
DISALLOWED_MAJOR_ROLES = {
    'Other',   # Do not allow catch-all "Other"
}

VALID_ROLES = []
for r in raw_roles:
    # Skip disallowed roles entirely
    if r in DISALLOWED_MAJOR_ROLES:
        continue
    # Apply replacements (e.g., Technologist -> Specialist)
    r_canon = MAJOR_REPLACEMENTS.get(r, r)
    if r_canon not in VALID_ROLES:
        VALID_ROLES.append(r_canon)

print(f"✅ Found {len(VALID_ROLES)} MNPS roles after cleaning/replacements")

# Enhanced closed sets for major and minor role groups
MAJOR_ALLOWED = VALID_ROLES
MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Enhanced specialist fallback patterns with Problem Role Cheat Sheet logic
SPECIALIST_FALLBACKS = [
    # Teacher patterns - classroom instruction, curriculum, students (Strong signal)
    ('Teacher', r'\b(teacher|classroom|lesson|instruction|teaching|curriculum|students|educational|academic|teach)\b'),
    # Coach patterns - instructional support, mentoring, professional development for staff (Strong signal)
    ('Coach', r'\b(coach|instructional coach|plc|model lessons?|co-teach|mentor|professional development|instructional support|facilitates professional learning)\b'),
    # Analyst patterns - data analysis, research, evaluation, statistical work (Strong signal)
    ('Analyst', r'\b(analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports|data collection|data reporting)\b'),
    # Technician patterns - hands-on technical work, equipment maintenance/troubleshooting (Strong signal)
    ('Technician', r'\b(technical|repair|troubleshoot|install|equipment|hands-on|hardware|software|systems|diagnose|fix|maintain|AV support|IT support)\b'),
    # Accountant patterns - financial, accounting, bookkeeping, payroll (Strong signal)
    ('Accountant', r'\b(accounting|financial|bookkeeping|audit|payroll|finance|accounts payable|accounts receivable|fiscal|budget tracking|compensation analysis)\b'),
    # Architect patterns - building/construction vs technology (Strong signal if primary focus)
    ('Architect (Facility-Focused)', r'\b(building|construction|facility|architectural|design|space planning|renovation|infrastructure|project management|capital improvement)\b'),
    ('Architect (Technology-Focused)', r'\b(system|software|technology|IT|database|network|programming|technical architecture)\b'),
    # Counselor patterns - guidance, therapy, mental health (Strong signal)
    ('Counselor', r'\b(counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological|student services)\b'),
    # Clerical patterns - administrative, office work, records (Strong signal)
    ('Clerical Support', r'\b(clerk|clerical|records|data entry|office support|administrative|filing|correspondence|reception|scheduling)\b'),
]

# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    """
    Discourage 'Specialist' only when there is strong evidence for a more specific role.
    Requires multiple keywords or role-defining language to prevent over-correction.
    """
    if proposed_major != 'Specialist':
        return proposed_major

    t = (text or '').lower()

    # Only reclassify if there's STRONG evidence of a better fit
    # Check for TEACHER (direct instruction)
    if any(kw in t for kw in ['teach', 'instruction', 'lesson', 'curriculum', 'classroom']):
        return 'Teacher'

    # Check for COACH (instructional adult learning)
    if 'instructional' in t and any(kw in t for kw in ['coach', 'mentor', 'plc', 'co-teach', 'professional development']):
        return 'Coach'

    # Check for ANALYST (statistical/data analysis focus)
    if any(kw in t for kw in ['statistical', 'quantitative analysis', 'regression', 'modeling']) or \
       (t.count('analyze') >= 2 and 'data' in t):
        return 'Analyst'

    # Check for TECHNICIAN (hands-on hardware/systems)
    if any(kw in t for kw in ['troubleshoot hardware', 'repair equipment', 'install systems']):
        return 'Technician'

    # Check for ACCOUNTANT (financial accounting tasks)
    if any(kw in t for kw in ['bookkeeping', 'audit', 'accounts payable', 'general ledger']):
        return 'Accountant'

    # Otherwise, keep as Specialist
    return 'Specialist'

def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    """Distinguish between Supervisor and Manager based on education requirements.
    Supervisor: Primarily manages people, no post-high school education required
    Manager: Does more than manage people, requires minimum associates degree
    """
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Check for education requirements
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)
    # Check for broader responsibilities beyond people management
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)
    # If has degree requirement or broader responsibilities, likely Manager
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'
    # If primarily people management without degree requirements, likely Supervisor
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'
    return proposed_major

def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    """Refine distinctions between Coordinator, Coach, and Manager based on Problem Role Cheat Sheet."""
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Coach patterns - instructional support, mentoring, professional development
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'
    # Manager patterns - strategic planning, policy, budget, supervision
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'
    # Coordinator patterns - coordination, organization, facilitation
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'
    return proposed_major

def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    """Fix minor sub-grouping for executive roles - rarely "Lead", usually "I", "II", or "III"."""
    if major_role not in EXECUTIVE_ROLES:
        return minor_role
    # If it's an executive role and currently "Lead", downgrade to "III" or "II"
    if minor_role == 'Lead':
        # Check if it's a very senior executive role that might warrant "III"
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'
    return minor_role

print("✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined")

# ==== 4) Build comprehensive KSACs text from all MNPS resources ====

In [ ]:
def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n"
    
    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()
    
    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)
    
    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")
    
    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)
    
    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")
    
    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)
    
    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")
    
    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")

# ==== 5) Enhanced Zero Shot Prompt with Problem Role Cheat Sheet Guidelines ====

In [ ]:
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.
Process:
- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Group jobs that have similar functions, responsibilities, and requirements, regardless of their job titles.
- Use the attached reference sources (Ground Truth Masterfile, MNPS Roles, MNPS KSACs) to ensure alignment with MNPS standards and classifications.
- Focus on the actual work being performed, not the job title, to create meaningful and accurate groupings.
- Ensure that each grouping reflects the true nature of the work and aligns with MNPS role classifications and competency frameworks.
- Provide clear justification for each grouping decision based on the job attributes and MNPS standards.
- Never justify classifications based on job titles - only use job attributes and MNPS standards.

IMPORTANT CLASSIFICATION GUIDELINES (Based on Problem Role Cheat Sheet):

ROLE DISTINCTIONS:
- **Technician vs Specialist vs Analyst**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting
  * Specialist: Specialized knowledge in specific domain, but prefer more specific roles when possible
  * Analyst: Data analysis, research, evaluation, assessment, statistical work, reporting

- **Coordinator vs Coach vs Manager**:
  * Coordinator: Coordination, organization, facilitation, liaison work, program coordination
  * Coach: Instructional support, mentoring, professional development, co-teaching, PLC facilitation
  * Manager: Strategic planning, policy development, budget oversight, supervision, management

- **Supervisor vs Manager**:
  * Supervisor: Primarily manages people, no post-high school education required
  * Manager: Does more than manage people, requires minimum associates degree

- **Architect Roles**:
  * Architect (Facility-Focused): Building/construction/space planning/renovation/infrastructure
  * Architect (Technology-Focused): System/software/IT/database/network/programming

MINOR SUB-GROUP GUIDELINES:
- **Executive Roles** (Coordinator, Principal, Director, Manager): Rarely "Lead", usually "I", "II", or "III"
- **"Lead"** should be reserved for non-executive roles that lead teams or projects
- **"III"** for very advanced KSACs and senior-level expertise
- **"II"** for intermediate complexity and responsibility
- **"I"** for entry-level or basic complexity

Output Requirements:
- Major Role Group: Choose from approved MNPS major role groupings
- Minor Sub Group: Use I, II, III, or Lead based on complexity and responsibility level (consider executive role guidelines)
- new_job_title: Should incorporate both major_role_group and minor_sub_group (e.g., "Accountant II", "Facility Coordinator II")
- Provide detailed justification based on job attributes and MNPS KSACs alignment that matches your selected role and level"""

print("✅ Enhanced zero shot prompt with Problem Role Cheat Sheet guidelines defined")

# NEW: Self-Consistency Prompt for Pass 2
self_consistency_prompt = \
"""You previously classified a job description and provided a justification. Now, review your own output for internal consistency.

**Instructions:**
- Compare your stated `major_role_group`, `minor_sub_group`, and `new_job_title` with your `grouping_justification`.
- If the justification **does not logically support** the selected role or level, **correct the classification** to match the reasoning.
- If the justification **supports a different role** (e.g., justification describes coaching but role is "Coordinator"), update the role accordingly.
- **Do not change the justification**—only update the classification fields if they conflict with it.
- Use **only approved MNPS roles** and **minor levels (I, II, III, Lead)**.
- Ensure `new_job_title` reflects the corrected role and level.
- If already consistent, return the original values unchanged.

**Return your response as a JSON object with this exact structure:**
{
  "new_job_title": "...",
  "major_role_group": "...",
  "minor_sub_group": "...",
  "grouping_justification": "..."  // <-- DO NOT MODIFY THIS FIELD
}"""

print("✅ Self-consistency prompt for Pass 2 defined")

# NEW: Manager→Director Promotion Check Prompt for Pass 3
triple_check_prompt = """
You are performing a third-pass "promotion check" for MNPS job classification.
You will ONLY be asked to review roles that were previously classified as "Manager".
Your job is to decide whether the role should remain "Manager" or be elevated to "Director".

Use these MNPS-specific guidelines:

1. Scope of responsibility
   - Manager: Owns a program, team, or sub-unit within an office/department; scope is usually local or departmental.
   - Director: Owns an entire function, office, or district-wide program area (often multiple programs) with system-wide impact.

2. Strategy vs operations
   - Manager: Focuses on implementation, day-to-day operations, and executing strategy set by others.
   - Director: Develops or co-develops strategy, sets direction and priorities, and is responsible for long-term planning.

3. People leadership
   - Manager: Supervises individuals and small teams (specialists, coordinators, technicians).
   - Director: Leads managers and/or multiple teams; provides leadership for an office or functional area.

4. Decision rights, budget, and policy
   - Manager: Implements policies and manages part of a budget within limits set by others.
   - Director: Develops or significantly shapes policies and procedures; owns or co-owns budgets and resource allocation for their function.

5. Accountability and stakeholders
   - Manager: Accountable for performance of a team/program; works mainly with school staff, principals, and department peers.
   - Director: Accountable for district- or system-level outcomes; collaborates with Chiefs, Executive Leadership, and external agencies; often represents MNPS externally.

Upgrade to "Director" ONLY IF the job description clearly shows MOST of the Director characteristics above,
such as district-wide scope, strategy setting, policy/budget ownership, and leadership of other leaders or multiple teams.

If evidence is mixed or ambiguous, KEEP IT AS "Manager".

Important constraints:
- Ignore the original job title text; use duties, responsibilities, scope, and KSAC-related content.
- You may ONLY choose "Manager" or "Director" as major_role_group.
- Keep the minor_sub_group consistent with MNPS conventions (I, II, III, or Lead; executive roles rarely have "Lead").

Return a JSON object with:
{
  "new_job_title": "Updated descriptive title if you upgrade to Director, otherwise keep or lightly refine",
  "major_role_group": "Manager" or "Director",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Explain why you kept Manager or upgraded to Director, citing specific phrases from the job description."
}
"""

print("✅ Manager→Director promotion check prompt for Pass 3 defined")

# NEW: Technician→Skilled Laborer Review Prompt for Pass 4
skilled_laborer_check_prompt = """
You are performing a fourth-pass "Technician vs Skilled Laborer" review for MNPS job classification.
You will ONLY be asked to review roles that were previously classified as "Technician".
Your job is to decide whether the role should remain "Technician" or be reclassified as "Skilled Laborer".

Use these MNPS-specific guidelines:

**Technician:**
- Focus on technical systems work requiring diagnostics and specialized technical knowledge
- HVAC systems (heating, ventilation, air conditioning) - technical diagnostics and system troubleshooting
- Network infrastructure, IT systems, audio-visual systems
- Automotive/vehicle diagnostics and repair
- Behavioral health interventions (e.g., Registered Behavior Technician with specialized training in ABA)
- Equipment troubleshooting requiring technical diagnostics and specialized certifications
- Installation and configuration of complex technical systems
- Examples: HVAC Technician, IT Technician, Audio-Visual Technician, Automotive Technician, Behavior Technician

**Skilled Laborer:**
- Focus on traditional trade-based work (plumbing, electrical, carpentry, painting, grounds)
- Physical work using hand tools and power tools to build, move, repair, or maintain physical environment
- Installation, assembly, and repair of physical structures and fixtures (not complex technical systems)
- Examples of trade work: plumbing (pipes, fixtures, drains), electrical wiring and fixtures, carpentry (building/repairing structures), painting, grounds maintenance, landscaping
- Moving furniture, loading/unloading equipment, warehouse work
- Building maintenance requiring general handyman skills rather than specialized technical diagnostics

**Key Distinguishers:**
- Technician = Systems + Diagnostics + Technical certifications
- Skilled Laborer = Trades + Physical work + General maintenance

Reclassify to "Skilled Laborer" ONLY IF the job description clearly shows traditional trade work as the PRIMARY focus.

If evidence is mixed or ambiguous, KEEP IT AS "Technician".

Important constraints:
- Ignore the original job title text; use duties, responsibilities, and required skills.
- You may ONLY choose "Technician" or "Skilled Laborer" as major_role_group.
- Keep the minor_sub_group consistent with MNPS conventions (I, II, III, or Lead).

Return a JSON object with:
{
  "new_job_title": "Updated descriptive title if you reclassify to Skilled Laborer, otherwise keep or lightly refine",
  "major_role_group": "Technician" or "Skilled Laborer",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Explain why you kept Technician or reclassified to Skilled Laborer, citing specific duties and skills from the job description."
}
"""

print("✅ Technician→Skilled Laborer review prompt for Pass 4 defined")

# NEW: Fifth Pass - Manager, Coordinator, Coach Review Prompt for Pass 5
fifth_pass_prompt = """
You are performing a fifth-pass review for MNPS job classification.
You will ONLY be asked to review roles that were previously classified as "Manager", "Coordinator", or "Coach".
Your job is to refine the classification to ensure the best fit among these three roles.

Use these MNPS-specific guidelines:

**Manager:**
- Strategic planning, policy development, budget oversight
- Supervision of staff (direct reports)
- Program or department leadership with decision-making authority
- Typically requires minimum associates degree
- Focuses on operations, implementation, and team management
- Examples: IT Manager, Facilities Manager, HR Manager, Operations Manager

**Coordinator:**
- Coordination, organization, facilitation, liaison work
- Program coordination without direct supervision of staff
- Project management, scheduling, logistics
- Typically does not require extensive education requirements
- Focuses on organizing and facilitating processes and programs
- Examples: Event Coordinator, Program Coordinator, Administrative Coordinator

**Coach:**
- Instructional support, mentoring, professional development for STAFF (not students)
- Co-teaching, modeling lessons, PLC facilitation
- Adult learning and instructional improvement focus
- Requires strong instructional expertise and often teaching experience
- Works WITH teachers/staff to improve instructional practice
- Examples: Instructional Coach, Literacy Coach, Math Coach

**Key Distinguishers:**
- Manager = Leadership + Supervision + Strategy
- Coordinator = Organization + Facilitation + Logistics (no direct supervision)
- Coach = Instructional support + Mentoring + Professional development for staff

Reclassify ONLY IF there is clear evidence that a different role is a better fit.

If evidence is mixed or ambiguous, KEEP THE ORIGINAL CLASSIFICATION.

Important constraints:
- Ignore the original job title text; use duties, responsibilities, and required skills.
- You may ONLY choose "Manager", "Coordinator", or "Coach" as major_role_group.
- Keep the minor_sub_group consistent with MNPS conventions (I, II, III; rarely "Lead" for these executive-adjacent roles).

Return a JSON object with:
{
  "new_job_title": "Updated descriptive title if you reclassify, otherwise keep or lightly refine",
  "major_role_group": "Manager", "Coordinator", or "Coach",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Explain why you kept or changed the classification, citing specific duties and responsibilities from the job description."
}
"""

print("✅ Fifth pass Manager/Coordinator/Coach review prompt for Pass 5 defined")

# ==== 6) OpenAI Client Setup ====

In [ ]:
# Set up OpenAI API key
import os
from getpass import getpass

# Check if API key is already in environment
if 'OPENAI_API_KEY' not in os.environ:
    print("OpenAI API key not found in environment.")
    api_key = getpass("Please enter your OpenAI API key: ")
    os.environ['OPENAI_API_KEY'] = api_key
else:
    print("✅ OpenAI API key found in environment")

# Initialize OpenAI client
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# Model to use
MODEL = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized with model: {MODEL}")

# ==== 7) Five-Pass Classification Pipeline ====

In [ ]:
def call_llm_with_retry(messages, max_retries=3, temperature=0.3):
    """Call OpenAI API with retry logic and rate limiting."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=temperature,
                response_format={"type": "json_object"}
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"⚠️  API call failed (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) + random.random()
                print(f"   Waiting {wait_time:.1f}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"   Max retries reached. Returning error response.")
                return json.dumps({
                    "new_job_title": "ERROR",
                    "major_role_group": "ERROR",
                    "minor_sub_group": "ERROR",
                    "grouping_justification": f"API call failed: {e}"
                })

def parse_llm_response(response_text):
    """Parse LLM JSON response with error handling."""
    try:
        return json.loads(response_text)
    except json.JSONDecodeError as e:
        print(f"⚠️  JSON parse error: {e}")
        print(f"   Response text: {response_text[:200]}...")
        return {
            "new_job_title": "PARSE_ERROR",
            "major_role_group": "PARSE_ERROR",
            "minor_sub_group": "PARSE_ERROR",
            "grouping_justification": f"Failed to parse JSON response"
        }

def classify_job_five_pass(job_text, row_index):
    """
    Five-pass classification pipeline:
    Pass 1: Initial LLM classification
    Pass 2: Self-consistency check
    Pass 3: Manager→Director promotion check
    Pass 4: Technician→Skilled Laborer review
    Pass 5: Manager/Coordinator/Coach refinement
    """
    
    print(f"\n{'='*70}")
    print(f"Processing Job {row_index}")
    print(f"{'='*70}")
    
    # PASS 1: Initial Classification
    print("\n🔍 PASS 1: Initial LLM Classification")
    pass1_messages = [
        {"role": "system", "content": zero_shot_prompt + "\n\n" + KSACS_TEXT},
        {"role": "user", "content": f"Classify this job based ONLY on its attributes (ignore title):\n\n{job_text}\n\nReturn JSON with: new_job_title, major_role_group, minor_sub_group, grouping_justification"}
    ]
    
    pass1_response = call_llm_with_retry(pass1_messages)
    pass1_result = parse_llm_response(pass1_response)
    
    print(f"   Initial Classification: {pass1_result['major_role_group']} {pass1_result['minor_sub_group']}")
    print(f"   Job Title: {pass1_result['new_job_title']}")
    
    # PASS 2: Self-Consistency Check
    print("\n🔍 PASS 2: Self-Consistency Check")
    pass2_messages = [
        {"role": "system", "content": self_consistency_prompt},
        {"role": "user", "content": f"Review this classification for consistency:\n\nOriginal Job Attributes:\n{job_text}\n\nYour Previous Classification:\n{json.dumps(pass1_result, indent=2)}\n\nCorrect any inconsistencies between the justification and the classification."}
    ]
    
    pass2_response = call_llm_with_retry(pass2_messages)
    pass2_result = parse_llm_response(pass2_response)
    
    if pass2_result['major_role_group'] != pass1_result['major_role_group']:
        print(f"   ⚠️  Role changed: {pass1_result['major_role_group']} → {pass2_result['major_role_group']}")
    else:
        print(f"   ✅ Classification consistent: {pass2_result['major_role_group']}")
    
    current_result = pass2_result
    
    # PASS 3: Manager→Director Promotion Check
    if current_result['major_role_group'] == 'Manager':
        print("\n🔍 PASS 3: Manager→Director Promotion Check")
        pass3_messages = [
            {"role": "system", "content": triple_check_prompt},
            {"role": "user", "content": f"Review if this Manager role should be promoted to Director:\n\nJob Attributes:\n{job_text}\n\nCurrent Classification:\n{json.dumps(current_result, indent=2)}"}
        ]
        
        pass3_response = call_llm_with_retry(pass3_messages)
        pass3_result = parse_llm_response(pass3_response)
        
        if pass3_result['major_role_group'] == 'Director':
            print(f"   ⬆️  PROMOTED: Manager → Director")
            current_result = pass3_result
        else:
            print(f"   ✅ Remains: Manager")
    else:
        print(f"\n⏭️  PASS 3: Skipped (not Manager)")
    
    # PASS 4: Technician→Skilled Laborer Review
    if current_result['major_role_group'] == 'Technician':
        print("\n🔍 PASS 4: Technician→Skilled Laborer Review")
        pass4_messages = [
            {"role": "system", "content": skilled_laborer_check_prompt},
            {"role": "user", "content": f"Review if this Technician role should be reclassified as Skilled Laborer:\n\nJob Attributes:\n{job_text}\n\nCurrent Classification:\n{json.dumps(current_result, indent=2)}"}
        ]
        
        pass4_response = call_llm_with_retry(pass4_messages)
        pass4_result = parse_llm_response(pass4_response)
        
        if pass4_result['major_role_group'] == 'Skilled Laborer':
            print(f"   🔄 RECLASSIFIED: Technician → Skilled Laborer")
            current_result = pass4_result
        else:
            print(f"   ✅ Remains: Technician")
    else:
        print(f"\n⏭️  PASS 4: Skipped (not Technician)")
    
    # PASS 5: Manager/Coordinator/Coach Refinement
    if current_result['major_role_group'] in ['Manager', 'Coordinator', 'Coach']:
        print("\n🔍 PASS 5: Manager/Coordinator/Coach Refinement")
        pass5_messages = [
            {"role": "system", "content": fifth_pass_prompt},
            {"role": "user", "content": f"Refine this classification among Manager/Coordinator/Coach:\n\nJob Attributes:\n{job_text}\n\nCurrent Classification:\n{json.dumps(current_result, indent=2)}"}
        ]
        
        pass5_response = call_llm_with_retry(pass5_messages)
        pass5_result = parse_llm_response(pass5_response)
        
        if pass5_result['major_role_group'] != current_result['major_role_group']:
            print(f"   🔄 REFINED: {current_result['major_role_group']} → {pass5_result['major_role_group']}")
            current_result = pass5_result
        else:
            print(f"   ✅ Remains: {current_result['major_role_group']}")
    else:
        print(f"\n⏭️  PASS 5: Skipped (not Manager/Coordinator/Coach)")
    
    # Apply post-processing corrections
    print("\n🔧 Applying Post-Processing Corrections")
    
    # Normalize minor sub-group
    current_result['minor_sub_group'] = normalize_minor(current_result['minor_sub_group'])
    
    # Apply specialist discouragement
    original_major = current_result['major_role_group']
    current_result['major_role_group'] = discourage_specialist(job_text, current_result['major_role_group'])
    if current_result['major_role_group'] != original_major:
        print(f"   📝 Specialist → {current_result['major_role_group']}")
    
    # Apply Supervisor/Manager distinction
    original_major = current_result['major_role_group']
    current_result['major_role_group'] = distinguish_supervisor_manager(job_text, current_result['major_role_group'])
    if current_result['major_role_group'] != original_major:
        print(f"   📝 {original_major} → {current_result['major_role_group']}")
    
    # Apply Coordinator/Coach/Manager refinement
    original_major = current_result['major_role_group']
    current_result['major_role_group'] = refine_coordinator_coach_manager(job_text, current_result['major_role_group'])
    if current_result['major_role_group'] != original_major:
        print(f"   📝 {original_major} → {current_result['major_role_group']}")
    
    # Fix executive minor sub-grouping
    original_minor = current_result['minor_sub_group']
    current_result['minor_sub_group'] = fix_executive_minor_sub_grouping(
        current_result['major_role_group'], 
        current_result['minor_sub_group']
    )
    if current_result['minor_sub_group'] != original_minor:
        print(f"   📝 Minor: {original_minor} → {current_result['minor_sub_group']}")
    
    # Update job title to match final classification
    current_result['new_job_title'] = f"{current_result['major_role_group']} {current_result['minor_sub_group']}"
    
    print(f"\n✅ FINAL CLASSIFICATION: {current_result['new_job_title']}")
    print(f"{'='*70}\n")
    
    # Add metadata
    current_result['source_row_index'] = row_index
    current_result['timestamp'] = dt.datetime.now().isoformat()
    
    return current_result

print("✅ Five-pass classification pipeline defined")

# ==== 8) Run Classification on All Jobs ====

In [ ]:
# Run classification on all jobs
print(f"\n🚀 Starting five-pass classification for {len(df)} jobs...\n")

results = []

for idx in range(len(df)):
    job_text = text.iloc[idx]
    
    try:
        result = classify_job_five_pass(job_text, idx)
        results.append(result)
        
        # Save intermediate results every 10 jobs
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_output = OUTPUTS_DIR / f"classifications_temp_{idx+1}.csv"
            temp_df.to_csv(temp_output, index=False)
            print(f"\n💾 Saved intermediate results: {temp_output}")
        
        # Rate limiting - wait between API calls
        time.sleep(0.5)
        
    except Exception as e:
        print(f"\n❌ Error processing job {idx}: {e}")
        results.append({
            "source_row_index": idx,
            "new_job_title": "ERROR",
            "major_role_group": "ERROR",
            "minor_sub_group": "ERROR",
            "grouping_justification": f"Processing error: {e}",
            "timestamp": dt.datetime.now().isoformat()
        })

# Create final results DataFrame
results_df = pd.DataFrame(results)

# Add original job information
results_df = results_df.merge(
    df[['Position Summary', 'Essential Functions', 'Work Experience', 'Education']],
    left_on='source_row_index',
    right_index=True,
    how='left'
)

# Save final results
final_output = OUTPUTS_DIR / "Job_Classifications_Batch.csv"
results_df.to_csv(final_output, index=False)

print(f"\n✅ Classification complete!")
print(f"📁 Final results saved to: {final_output}")
print(f"\n📊 Summary:")
print(f"   Total jobs processed: {len(results_df)}")
print(f"   Successful classifications: {len(results_df[results_df['major_role_group'] != 'ERROR'])}")
print(f"   Errors: {len(results_df[results_df['major_role_group'] == 'ERROR'])}")

# Display role distribution
print(f"\n📊 Role Distribution:")
role_counts = results_df['major_role_group'].value_counts()
for role, count in role_counts.items():
    print(f"   {role}: {count}")

# ==== 9) Analysis and Visualization ====

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('MNPS Job Classification Analysis - v7.5.5', fontsize=16, fontweight='bold')

# 1. Major Role Group Distribution
role_counts = results_df['major_role_group'].value_counts()
axes[0, 0].barh(range(len(role_counts)), role_counts.values, color='steelblue')
axes[0, 0].set_yticks(range(len(role_counts)))
axes[0, 0].set_yticklabels(role_counts.index)
axes[0, 0].set_xlabel('Count')
axes[0, 0].set_title('Major Role Group Distribution')
axes[0, 0].grid(True, alpha=0.3)

# 2. Minor Sub Group Distribution
minor_counts = results_df['minor_sub_group'].value_counts()
axes[0, 1].bar(range(len(minor_counts)), minor_counts.values, color='coral')
axes[0, 1].set_xticks(range(len(minor_counts)))
axes[0, 1].set_xticklabels(minor_counts.index)
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Minor Sub Group Distribution')
axes[0, 1].grid(True, alpha=0.3)

# 3. Top 10 Job Titles
top_titles = results_df['new_job_title'].value_counts().head(10)
axes[1, 0].barh(range(len(top_titles)), top_titles.values, color='lightgreen')
axes[1, 0].set_yticks(range(len(top_titles)))
axes[1, 0].set_yticklabels(top_titles.index, fontsize=9)
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_title('Top 10 Job Titles')
axes[1, 0].grid(True, alpha=0.3)

# 4. Role × Level Heatmap
role_level_crosstab = pd.crosstab(results_df['major_role_group'], results_df['minor_sub_group'])
sns.heatmap(role_level_crosstab, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1, 1], cbar_kws={'label': 'Count'})
axes[1, 1].set_title('Role × Level Distribution')
axes[1, 1].set_xlabel('Minor Sub Group')
axes[1, 1].set_ylabel('Major Role Group')

plt.tight_layout()

# Save visualization
viz_path = OUTPUTS_DIR / "classification_analysis.png"
plt.savefig(viz_path, dpi=300, bbox_inches='tight')
print(f"\n📊 Visualization saved to: {viz_path}")

plt.show()

# ==== 10) Generate Summary Report ====

In [ ]:
# Generate summary report
report_text = f"""MNPS JOB CLASSIFICATION SUMMARY REPORT
Version 7.5.5 - Five-Pass Classification System
Generated: {dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
{'='*80}

OVERVIEW:
This report summarizes the results of the five-pass job classification system
that processes {len(results_df)} job descriptions using GPT-4o with enhanced
role distinction logic and self-consistency checking.

CLASSIFICATION STATISTICS:
{'='*80}

Total Jobs Processed: {len(results_df)}
Successful Classifications: {len(results_df[results_df['major_role_group'] != 'ERROR'])}
Errors: {len(results_df[results_df['major_role_group'] == 'ERROR'])}

MAJOR ROLE GROUP DISTRIBUTION:
{'='*80}
"""

for role, count in role_counts.items():
    pct = (count / len(results_df)) * 100
    report_text += f"{role:.<40} {count:>5} ({pct:>5.1f}%)\n"

report_text += f"""
MINOR SUB GROUP DISTRIBUTION:
{'='*80}
"""

for level, count in minor_counts.items():
    pct = (count / len(results_df)) * 100
    report_text += f"{level:.<40} {count:>5} ({pct:>5.1f}%)\n"

report_text += f"""
KEY IMPROVEMENTS IN v7.5.5:
{'='*80}

1. Five-Pass Classification System:
   - Pass 1: Initial LLM classification using full MNPS context
   - Pass 2: Self-consistency check to align justification with classification
   - Pass 3: Manager→Director promotion check for strategic/executive scope
   - Pass 4: Technician→Skilled Laborer review for trade-based work
   - Pass 5: Manager/Coordinator/Coach refinement for better distinction

2. Enhanced Role Distinctions:
   - Technician vs Skilled Laborer (systems work vs trade work)
   - Specialist underusage addressed with stronger fallback logic
   - Coordinator vs Coach vs Manager (organization vs mentoring vs leadership)
   - Supervisor vs Manager (education requirements and scope)

3. Problem Role Cheat Sheet Integration:
   - Explicit guidelines for confusing role pairs
   - Minor sub-group guidelines for executive roles
   - Attribute-only classification (job title ignored)

4. Quality Improvements:
   - Self-consistency checking reduces justification mismatches
   - Targeted review passes for high-confusion roles
   - Post-processing corrections for edge cases

OUTPUT FILES:
{'='*80}

Main Results: {OUTPUTS_DIR / 'Job_Classifications_Batch.csv'}
Visualization: {OUTPUTS_DIR / 'classification_analysis.png'}
This Report: {OUTPUTS_DIR / 'summary_report.txt'}

{'='*80}
END OF REPORT
"""

# Save report
report_path = OUTPUTS_DIR / "summary_report.txt"
with open(report_path, 'w') as f:
    f.write(report_text)

print(f"\n📄 Summary report saved to: {report_path}")
print("\n" + report_text)

# ==== 11) Complete - All Outputs Saved ====

In [ ]:
print("\n" + "="*80)
print("CLASSIFICATION COMPLETE - ALL FILES SAVED")
print("="*80)
print(f"\n📁 All outputs saved to: {OUTPUTS_DIR}")
print(f"\nGenerated Files:")
print(f"  1. Job_Classifications_Batch.csv - Final classification results")
print(f"  2. classification_analysis.png - Visual analysis")
print(f"  3. summary_report.txt - Detailed summary report")
print(f"\n✅ Run timestamp: {timestamp}")
print(f"✅ Total jobs processed: {len(results_df)}")
print(f"✅ Success rate: {(len(results_df[results_df['major_role_group'] != 'ERROR']) / len(results_df) * 100):.1f}%")
print("\n" + "="*80)